#### this script assign role of author based on the seqno column to all publications including no international and domestic, seqno==1 is assigned to F, last seqno is assigned to L and the middle no assigned to M. 

In [1]:
import pandas as pd

In [2]:
author_path="../../data/dropbox/LaborDivision/Data/cleanedraw/authors.pkl"
pubs_path='../../data/dropbox/LaborDivision/Data/deriveddata/pub_detail.pkl'

In [3]:
author_order_path='../../data/dropbox/LaborDivision/Data/deriveddata/authors_ordered.pkl'

In [4]:
author_df=pd.read_pickle(author_path)

In [6]:
author_df.head()

,id,year,dis,full,first,last,seqno,addrno,cntry,Code,cluster_id
0,29768068,2008,56,"Miron, Veronique E.",Veronique E.,Miron,1,1,Canada,CAN,1306275
1,29768068,2008,56,"Hall, Jeffery A.",Jeffery A.,Hall,2,3,Canada,CAN,1680670
2,29768068,2008,56,"Antel, Jack P.",Jack P.,Antel,5,1,Canada,CAN,28261869
3,29768070,2008,51,"Shen, Jiao",Jiao,Shen,1,1,Japan,JPN,20009795
4,29768070,2008,51,"Tanida, Mamoru",Mamoru,Tanida,2,1,Japan,JPN,29411586


In [8]:
pub_df=pd.read_pickle(pubs_path)
pub_df.head()

,id,year,dis,alpha,teamsize,int
0,38579295,2012,1,no,4,0
1,38579296,2012,120,no,4,0
2,38579297,2012,84,no,4,1
3,38579302,2012,55,no,4,0
4,38579305,2012,43,no,4,0


In [9]:
pub_df['id'].unique().tolist())

18684905

In [12]:
len(author_df['id'].unique().tolist())

18684905

In [10]:
nanid=pub_df[(pub_df.alpha=='yes')|(pub_df.teamsize<3)]['id'].values
null_df=author_df[author_df.id.isin(nanid)]

In [11]:
null_df['role']='null'

<ipython-input-11-9f33e8f0ead9>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  null_df['role']='null'


In [13]:
#select the pubs that need to assign roles which include pubs with at least 3 authors and not-alphabetically ordered
ids=pub_df[(pub_df.alpha=='no')&(pub_df.teamsize>2)]['id'].values
author_df=author_df[author_df.id.isin(ids)]

In [14]:
first_df= author_df[author_df.seqno==1]
first_df['role']='F'

<ipython-input-14-3ddd93d09f41>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_df['role']='F'


In [15]:
author_df.head()

,id,year,dis,full,first,last,seqno,addrno,cntry,Code,cluster_id
0,29768068,2008,56,"Miron, Veronique E.",Veronique E.,Miron,1,1,Canada,CAN,1306275
1,29768068,2008,56,"Hall, Jeffery A.",Jeffery A.,Hall,2,3,Canada,CAN,1680670
2,29768068,2008,56,"Antel, Jack P.",Jack P.,Antel,5,1,Canada,CAN,28261869
3,29768070,2008,51,"Shen, Jiao",Jiao,Shen,1,1,Japan,JPN,20009795
4,29768070,2008,51,"Tanida, Mamoru",Mamoru,Tanida,2,1,Japan,JPN,29411586


In [16]:
author_df=author_df.sort_values(by=['id','seqno'])
last_df=author_df.drop_duplicates(subset=['id'],keep='last')
last_df['role']="L"

<ipython-input-16-7b2e47624e5f>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_df['role']="L"


In [19]:
f=author_df.duplicated(['id'],keep='first')# the first occurrence is marked as false
l=author_df.duplicated(['id'],keep='last')# the last occurrence is marked as false
middle_df=author_df[f&l]# middle authors are marked as true
middle_df['role']="M"

<ipython-input-19-57e6f1c092ab>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  middle_df['role']="M"


In [20]:
author_new=pd.concat([first_df,middle_df,last_df,null_df])

In [21]:
author_new=author_new.sort_values(by=['id','seqno'])

In [25]:
author_new = author_new.drop(columns=['full','first','last','seqno','addrno','cntry'])
author_new[['id','year','dis','cluster_id','role','Code']].to_pickle(author_order_path)